# 🔄 Iterators, Generators, Recursion & Async in Python

### 📖 The Journey
Most developers use `for` loops and `yield` daily without knowing the hidden machinery underneath. Today, we will deconstruct Python's iteration protocol from the C-level mental model all the way to modern `async/await`.

### 🗺️ Notebook Roadmap
```mermaid
flowchart TD

A["🔄 Iteration"] --> B["📦 Iterable"]
B --> C["🏃 Iterator"]
C --> D["⚙️ iter() & next()"]

D --> E["🔍 How For Loops Work"]
E --> F["🏗️ Build Custom Iterators"]
F --> G["🚀 Generators & yield"]

G --> H["💾 Memory Efficiency"]
H --> I["🔗 Generator Pipelines"]

I --> J["♻️ Recursion"]
J --> K["⚡ Memoization"]

K --> L["🎤 Interview Mastery"]
L --> M["🌐 Async/Await"]
```

## 🔄 1. What is Iteration?

**Iteration** is the general process of accessing items in a collection one after another.

Any time you use a loop (explicit like `for`/`while`, or implicit like list comprehensions) to go over a group of items, you are performing iteration.

### 🧠 Mental Model
Think of iteration like reading a book page by page. You don't read the whole book in a single microsecond; you process it sequentially, one page (item) at a time.

In [2]:
# ==========================================
# 🔄 BASIC ITERATION
# ==========================================
print("--- Iterating over a List ---")
numbers = [10, 20, 30, 40]
for num in numbers:
    print(f"Processing: {num}")

print("\n--- Iterating over a String ---")
for char in "Python":
    print(f"Character: {char}")

--- Iterating over a List ---
Processing: 10
Processing: 20
Processing: 30
Processing: 40

--- Iterating over a String ---
Character: P
Character: y
Character: t
Character: h
Character: o
Character: n


## 📦 2. What is an Iterable?

An **Iterable** is any object in Python that can be looped over. It is a collection of items that knows how to produce an **Iterator** when asked. It possesses an `__iter__()` method that yields an Iterator. Examples include lists, tuples, dictionaries, and strings.

### 📜 The Rule
To be an Iterable, an object MUST implement the `__iter__()` method. This method returns an Iterator.

### 🎬 Common Iterables
* `list`, `tuple`, `string`, `dict`, `set`
* `range()`, `open()` (files)
* Any custom class with an `__iter__()` method.

In [1]:
# ==========================================
# 📦 CHECKING FOR ITERABLES
# ==========================================

my_list = [1, 2, 3]
my_string = "Hello"

print(f"Is list an Iterable? {hasattr(my_list, '__iter__')}")
print(f"Is string an Iterable? {hasattr(my_string, '__iter__')}")

Is list an Iterable? True
Is string an Iterable? True


## 🏃‍♂️ 3. What is an Iterator?

An **Iterator** is the actual "engine" that performs the iteration. It is an object that maintains the **state** and knows how to fetch the *next* value. It must possess both an `__iter__()` method (returning itself) and a `__next__()` method.


### 🎬 The Netflix Bookmark Analogy
* **The Iterable** is the **DVD Box Set**. It contains all the episodes, but you can't just "play" the box. 
* **The Iterator** is the **DVD Player's Bookmark**. It remembers exactly which episode you are currently on. When you press "Next", it plays the next episode and updates the bookmark.

### 📜 The Rules
To be an Iterator, an object MUST implement:
1. `__iter__()`: Returns itself (so it can be used in `for` loops).
2. `__next__()`: Returns the next item, or raises `StopIteration` when exhausted.

### Why Iterators Matter
When building machine learning models or analyzing massive SQL dumps, you cannot load a 50GB dataset into standard RAM all at once.
Iterables (like Lists) use **Eager Evaluation** and store everything in memory. Iterators (like `range()`) use **Lazy Evaluation**, calculating and yielding only the next value on the fly, consuming almost zero memory.



## ⚔️ 4. Iterable vs Iterator

| Feature | Iterable (The Box Set) | Iterator (The Bookmark) |
| :--- | :--- | :--- |
| **Definition** | Collection of items. | Object that tracks state & produces values. |
| **Methods** | Has `__iter__()` | Has `__iter__()` AND `__next__()` |
| **Memory** | Stores all data in memory (usually). | Generates/holds one item at a time. |
| **Reusability** | Can be looped infinitely. | **One-time use.** Once exhausted, it's dead. |
| **Examples** | `list`, `str`, `dict` | `iter(list)`, `map()`, `zip()`, Generators |

> 💡 **Golden Rule:** Every Iterator is an Iterable. But NOT every Iterable is an Iterator.

In [ ]:
# ==========================================
# ⚔️ THE TRICK: Iterable vs Iterator
# ==========================================
L = [1, 2, 3]

print("--- The List (Iterable) ---")
print(f"Has __iter__? {'__iter__' in dir(L)}") # True
print(f"Has __next__? {'__next__' in dir(L)}") # False! Not an iterator.

print("\n--- The Iterator ---")
iter_L = iter(L)
print(f"Has __iter__? {'__iter__' in dir(iter_L)}") # True
print(f"Has __next__? {'__next__' in dir(iter_L)}") # True! It is an iterator.

## 🔑 5. The `iter()` Function

The `iter()` function is the bridge between an Iterable and an Iterator. If you pass an Iterator into the `iter()` function, it simply returns its own memory address.
When you call `iter(obj)`, Python asks the object: *"Give me your iterator."*

**Why?** Because the for loop blindly calls `iter()` on whatever you hand it. By defining `__iter__()` to return `self`, an Iterator satisfies the Iterable protocol and can be used directly in loops without crashing.

### 🧠 Memory Diagram
```text
List Object (Iterable)
┌─────────────────────┐
│ [1, 2, 3]           │
│ __iter__() ─────────┼──────┐
└─────────────────────┘      │
                             ▼
                     ┌─────────────────────┐
                     │ list_iterator       │
                     │ index = 0           │
                     │ __next__()          │
                     └─────────────────────┘
```

In [3]:
# ==========================================
# 🔑 iter() and next()
# ==========================================
colors = ["Red", "Green", "Blue"]

# 1. Get the iterator (Create the bookmark)
color_iter = iter(colors)

# 2. Fetch values manually using next()
print(next(color_iter))  # Red (index moves to 1)
print(next(color_iter))  # Green (index moves to 2)
print(next(color_iter))  # Blue (index moves to 3)

Red
Green
Blue


In [15]:
# To build an iterator from scratch, we need two classes.

class mera_range_iterator:
    """The Iterator (Maintains State)"""
    def __init__(self, iterable_obj):
        self.iterable = iterable_obj
    
    def __iter__(self):
        return self  # The confusing point rule!
    
    def __next__(self):
        if self.iterable.start >= self.iterable.end:
            raise StopIteration
            
        current = self.iterable.start
        self.iterable.start += 1
        return current

class mera_range:
    """The Iterable (Holds Data Params)"""
    def __init__(self, start, end):
        self.start = start
        self.end = end
        
    def __iter__(self):
        return mera_range_iterator(self)

for num in mera_range(1, 4):
    print("Custom Range output:", num)

Custom Range output: 1
Custom Range output: 2
Custom Range output: 3


## 🛑 6 & 7. The `next()` Function & `StopIteration`

The `next()` function asks the iterator for the next value. 
But what happens when we reach the end of the collection? 

The iterator doesn't return `None`. It raises a **`StopIteration`** exception. This is Python's way of signaling: *"There are no more items."*

In [4]:
# ==========================================
# 🛑 StopIteration
# ==========================================
colors = ["Red", "Green", "Blue"]
color_iter = iter(colors)

next(color_iter) # Red
next(color_iter) # Green
next(color_iter) # Blue

try:
    next(color_iter) # 💥 Boom!
except StopIteration:
    print("\n💥 StopIteration caught! The iterator is completely exhausted.")


💥 StopIteration caught! The iterator is completely exhausted.


## 🤯 8. How the `for` loop ACTUALLY works

Python doesn't have a native `for` loop at the C-level. The `for` loop is just **syntactic sugar** for a `while` loop that handles `StopIteration` automatically!

```python
for item in iterable:
    print(item)
```


```mermaid
flowchart TD

A["for item in iterable"]
--> B["iter(iterable)"]

B --> C["Iterator Created"]

C --> D["next(iterator)"]

D --> E{"Value Available?"}

E -->|Yes| F["Return Value"]
F --> G["Execute Loop Body"]
G --> D

E -->|No| H["Raise StopIteration"]

H --> I["Loop Ends Gracefully"]
```

```mermaid
sequenceDiagram
    participant User as Your Code
    participant Python as For Loop Engine
    participant Iterable as Iterable Object
    participant Iterator as Iterator Object

    User->>Python: for item in iterable
    Python->>Iterable: iter(iterable)
    Iterable-->>Python: Iterator Object

    loop Repeated Until Exhausted
        Python->>Iterator: next(iterator)
        Iterator-->>Python: Next Value
        Python->>User: Execute Loop Body
    end

    Python->>Iterator: next(iterator)
    Iterator-->>Python: StopIteration
    Python->>User: Exit Loop Gracefully
```

In [5]:
# ==========================================
# 🛠️ BUILDING OUR OWN FOR LOOP
# ==========================================
def mera_khudka_for_loop(iterable):
    """Replicates Python's for loop exactly"""
    
    # Step 1: Get the iterator
    iterator = iter(iterable)
    
    # Step 2: Infinite loop
    while True:
        try:
            # Step 3: Fetch next item
            item = next(iterator)
            print(item)  # Loop body
        except StopIteration:
            # Step 4: Break gracefully when exhausted
            break

print("Testing custom for loop:")
mera_khudka_for_loop([10, 20, 30])

Testing custom for loop:
10
20
30


## 🤔 9. The "Confusing Point"

If you pass an **Iterator** into the `iter()` function, what happens?
It just returns **itself**! The memory address remains exactly the same.

### 🧠 Why did Python design it this way?
Because the `for` loop protocol **blindly** calls `iter()` on whatever you pass to it. 
If you pass a list, it creates an iterator. If you pass an *already existing iterator*, Python needs to know it's already an iterator. By defining `__iter__` to return `self`, **Iterators satisfy the Iterable protocol**, allowing them to be used in `for` loops seamlessly.

In [6]:
# ==========================================
# 🕵️‍♂️ THE CONFUSING POINT PROOF
# ==========================================
num = [1, 2, 3]

# Get iterator from list
iter_obj1 = iter(num)
print(f"Address of iter_obj1: {id(iter_obj1)}")

# Pass iterator back into iter()
iter_obj2 = iter(iter_obj1)
print(f"Address of iter_obj2: {id(iter_obj2)}")

print(f"\n✅ Are they the EXACT same object? {iter_obj1 is iter_obj2}")

Address of iter_obj1: 1947193538304
Address of iter_obj2: 1947193538304

✅ Are they the EXACT same object? True


## 🏗️ 10 & 11. Custom Iterator Class & Our Own `range()`

To create a custom iterator from scratch using OOP, we need **Two Classes**:
1. **The Iterable Class:** Contains the data and defines `__iter__()`.
2. **The Iterator Class:** Contains the state and defines `__next__()`.

In [7]:
# ==========================================
# 🏗️ OOP ITERATOR: Custom Range
# ==========================================
class mera_range:
    """The Iterable"""
    def __init__(self, start, end):
        self.start = start
        self.end = end
        
    def __iter__(self):
        return mera_range_iterator(self)

class mera_range_iterator:
    """The Iterator"""
    def __init__(self, iterable_obj):
        self.current = iterable_obj.start
        self.end = iterable_obj.end
    
    def __iter__(self):
        return self  # Crucial for the "Confusing Point" rule!
    
    def __next__(self):
        if self.current >= self.end:
            raise StopIteration
        
        current_val = self.current
        self.current += 1  # Update state
        return current_val

print("Using custom range in a for loop:")
for i in mera_range(1, 6):
    print(i, end=" | ")

Using custom range in a for loop:
1 | 2 | 3 | 4 | 5 | 

## 🚀 12. Generators: The Pythonic Shortcut

Writing two separate OOP classes (`__iter__` and `__next__`) just to create an iterator is tedious. Enter **Generators**. Generators are functions that use the `yield` keyword to automatically construct an iterator under the hood.

A Generator is a simple, elegant way to create Iterators using **Functions** and the **`yield`** keyword. Python automatically handles the `__iter__`, `__next__`, and `StopIteration` boilerplate for you.

| Feature | `return` | `yield` |
| --- | --- | --- |
| **Execution Behavior** | Terminates the function entirely. | Pauses the function and saves the local state. |
| **Memory Load** | Returns all data sequentially into RAM. | Yields one single item at a time. |
| **Use Case** | Standard calculations. | Streaming infinite data or large image batches. |

In [8]:
# ==========================================
# 🚀 GENERATORS: The `yield` Keyword
# ==========================================
def gen_demo():
    yield "first statement"
    yield "second statement"
    yield "third statement"

# Calling the function DOES NOT execute it. It returns a Generator Object!
gen = gen_demo()
print(f"Type of gen: {type(gen)}")

print("\nFetching values using next():")
print(next(gen)) # Resumes, hits first yield, pauses
print(next(gen)) # Resumes, hits second yield, pauses
print(next(gen)) # Resumes, hits third yield, pauses

Type of gen: <class 'generator'>

Fetching values using next():
first statement
second statement
third statement


In [16]:
# 1. Generator Expression (Notice the parentheses instead of brackets)
gen_expr = (i**2 for i in range(1, 6))
print("Generator Expression:", list(gen_expr))

# 2. Chaining Generators (Creating a Pipeline)
def fibonacci_stream(nums):
    x, y = 0, 1
    for _ in range(nums):
        x, y = y, x + y
        yield x

def square_stream(nums):
    for num in nums:
        yield num ** 2

# The data flows through the pipeline one item at a time (O(1) memory)
pipeline_result = sum(square_stream(fibonacci_stream(10)))
print("Sum of squared Fibonacci numbers:", pipeline_result)

Generator Expression: [1, 4, 9, 16, 25]
Sum of squared Fibonacci numbers: 4895


## 🧠 13. `yield` vs `return`: What state is actually saved?

When a regular function hits `return`, it is destroyed. The local variables vanish.
When a generator hits `yield`, Python **freezes** the function. 

### 🥶 What exactly is saved in memory?
1. **The Instruction Pointer:** The exact line of code to resume from.
2. **Local Variables:** The values of all local variables at that exact moment.
3. **Internal State:** The generator's internal C-struct state.

> 💡 **Key Insight:** It does NOT save the global state, nor does it save the call stack of the caller. It only saves its own local frame.

In [9]:
# ==========================================
# 🥶 PROOF: State is Saved
# ==========================================
def counter_gen():
    count = 0
    while True:
        count += 1
        yield count

gen = counter_gen()
print(next(gen)) # 1 (count is saved as 1)
print(next(gen)) # 2 (count resumes from 1, becomes 2)
print(next(gen)) # 3 (count resumes from 2, becomes 3)

1
2
3


## 📊 14 & 15. Generator Expressions & Memory Efficiency

Just like List Comprehensions use `[]`, **Generator Expressions** use `()`.
They are incredibly memory efficient because they evaluate **lazily** (on-demand).

In [10]:
# ==========================================
# 📊 MEMORY BENCHMARKS
# ==========================================
import sys

# 1. List Comprehension (Eager Evaluation)
list_comp = [x**2 for x in range(1_000_000)]
print(f"📦 Memory of List (1M items): {sys.getsizeof(list_comp) / (1024*1024):.2f} MB")

# 2. Generator Expression (Lazy Evaluation)
gen_exp = (x**2 for x in range(1_000_000))
print(f"🏃 Memory of Gen (1M items): {sys.getsizeof(gen_exp)} Bytes")

print("\n✅ Conclusion: The generator takes the same tiny amount of memory regardless of size!")

📦 Memory of List (1M items): 8.06 MB
🏃 Memory of Gen (1M items): 200 Bytes

✅ Conclusion: The generator takes the same tiny amount of memory regardless of size!


In [11]:
# ==========================================
# 🌌 INFINITE STREAMS & PIPELINES
# ==========================================

# 1. Infinite Stream (Impossible with Lists)
def all_even():
    n = 0
    while True:  
        yield n
        n += 2

even_gen = all_even()
print("First 5 even numbers:")
for _ in range(5):
    print(next(even_gen), end=" ")

# 2. Generator Pipelines (Data Processing)
print("\n\nChaining Generators (Pipeline):")
def fibonacci_numbers(nums):
    x, y = 0, 1
    for _ in range(nums):
        x, y = y, x+y
        yield x

def square(nums):
    for num in nums:
        yield num**2

# Pipeline: Fibonacci -> Square -> Sum
# Memory used at any point: O(1)!
total_sum = sum(square(fibonacci_numbers(10)))
print(f"Sum of squared Fibonacci: {total_sum}")

First 5 even numbers:
0 2 4 6 8 

Chaining Generators (Pipeline):
Sum of squared Fibonacci: 4895


## 🔁 16. Recursion & Memoization

**Recursion** occurs when a function calls itself to solve a smaller instance of the identical problem. It requires two distinct components: 
1. **Base Case:** The strict stopping condition to prevent a Stack Overflow.
2. **Decomposition:** Breaking the problem into smaller subproblems.

### 🐌 The Problem: Exponential Time
Standard recursive Fibonacci runs in $O(2^n)$ because it recalculates the same subproblems repeatedly.

### 🚀 The Solution: Memoization
Memoization involves caching previously computed results in a dictionary, reducing the time complexity to linear time, or $O(n)$.


In [12]:
# ==========================================
# 🔁 RECURSION & MEMOIZATION
# ==========================================
import time

# 1. Naive Recursion (Slow)
def fib_naive(n):
    if n <= 1: return n
    return fib_naive(n-1) + fib_naive(n-2)

# 2. Memoized Recursion (Fast)
def fib_memo(n, cache={0:0, 1:1}):
    if n not in cache:
        cache[n] = fib_memo(n-1, cache) + fib_memo(n-2, cache)
    return cache[n]

# 3. Python's Built-in Memoization Decorator
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_pythonic(n):
    if n <= 1: return n
    return fib_pythonic(n-1) + fib_pythonic(n-2)

# Benchmark
n = 35
start = time.time()
fib_pythonic(n)
print(f"✅ Pythonic Memoized fib({n}) took: {time.time() - start:.6f} seconds")

✅ Pythonic Memoized fib(35) took: 0.000000 seconds


In [13]:
# ==========================================
# ⚡ ADVANCED RECURSION: POWERSET
# ==========================================
def powerset(elements):
    """Generate all subsets of a set using recursion"""
    if len(elements) == 0:
        return [[]]  # Base case: the empty set
    
    first = elements[0]
    rest = elements[1:]
    
    # Recursively get subsets without the first element
    subsets_without_first = powerset(rest)
    
    # Create subsets WITH the first element
    subsets_with_first = [[first] + subset for subset in subsets_without_first]
    
    return subsets_without_first + subsets_with_first

print("PowerSet of ['1', '2', '3']:")
print(powerset(['1', '2', '3']))

PowerSet of ['1', '2', '3']:
[[], ['3'], ['2'], ['2', '3'], ['1'], ['1', '3'], ['1', '2'], ['1', '2', '3']]


In [17]:
import time

def memoized_fib(m, cache):
    # Check if result is already calculated
    if m in cache:
        return cache[m]
    
    # Calculate and store in cache
    cache[m] = memoized_fib(m-1, cache) + memoized_fib(m-2, cache)
    return cache[m]

# Base cases stored in dictionary
fib_cache = {0: 1, 1: 1}

start_time = time.time()
print("500th Fibonacci:", memoized_fib(500, fib_cache))
print(f"Execution Time: {time.time() - start_time:.5f} seconds")

500th Fibonacci: 225591516161936330872512695036072072046011324913758190588638866418474627738686883405015987052796968498626
Execution Time: 0.00000 seconds


## 🎤 17. FAANG Interview Cheat Sheet

### 🟢 Beginner Level
**Q1: What is the difference between an Iterable and an Iterator?**
> **A:** An *Iterable* is a collection (like a list) that implements `__iter__()`. An *Iterator* is the stateful engine that implements `__next__()` and fetches items one by one.

**Q2: What happens when a `for` loop runs out of items?**
> **A:** The iterator raises a `StopIteration` exception. The `for` loop catches this silently and exits.

### 🟡 Intermediate Level
**Q3: What exactly does `yield` save in memory?**
> **A:** It saves the function's local frame: the instruction pointer (where to resume) and the local variables' state. It does not save the global state or caller's stack.

**Q4: Can you iterate over an Iterator twice?**
> **A:** No. Iterators are strictly stateful and single-use. Once `StopIteration` is triggered, the iterator is exhausted. You must create a brand-new iterator to loop again or you must call `iter()` on the original *Iterable* to get a fresh Iterator.

### 🔴 Advanced Level
**Q5: How does `iter(iterator)` work?**
> **A:** It returns the iterator itself (`self`). This allows iterators to be seamlessly passed into `for` loops, which blindly call `iter()` on their target.

**Q6: Explain Generator Pipelines and their memory complexity.**
> **A:** Generators evaluate lazily. When chained (e.g., `sum(square(fib(10)))`), data flows one item at a time through the pipeline. The memory complexity is $O(1)$, regardless of the dataset size.

**Q7: Why would you choose Iteration over Recursion?**
> **A:** Recursion is elegant for tree-like structures, but every recursive call adds a new layer to the system's Call Stack. Deep recursion causes Stack Overflow errors and consumes more memory, whereas iteration runs safely in a single stack frame.

**Q8: What is the difference between a list comprehension and a generator expression?**
> **A:** A list comprehension eagerly evaluates and stores the entire array in memory. A generator expression lazily evaluates, returning an iterator that computes values on demand, saving massive amounts of RAM.

## 🌌 18. The Missing Link: From Iterators to Async/Await

Most tutorials stop at Generators. But Python's modern concurrency model is a direct evolution of the Iterator protocol.

### 🧬 The Evolution

```mermaid
flowchart TD

A["Iterable"]
--> B["Iterator"]

B --> C["Iterator Protocol<br/>__iter__() + __next__()"]

C --> D["Generators<br/>yield"]

D --> E["Generator Expressions"]

E --> F["Coroutines<br/>send()"]

F --> G["yield from"]

G --> H["Async Generators"]

H --> I["async / await"]

I --> J["Concurrent Python Applications"]
```

### 🧠 How did we get to `async/await`?
1. **Iterators:** Can only *push* data out (`__next__`).
2. **Generators:** Can *push* data out (`yield`).
3. **Coroutines:** In Python 2.5, `yield` was upgraded. Generators could now *receive* data via `gen.send(value)`. This made them **Coroutines** (functions that can pause, resume, and exchange data).
4. **`yield from`:** Python 3.3 added `yield from` to delegate to sub-generators, cleaning up coroutine chains.
5. **`async/await`:** In Python 3.5, native coroutines were introduced. `async def` replaces generator-based coroutines, and `await` replaces `yield from`. Under the hood, they still rely on an event loop calling `__next__` and `send()`!
```



In [14]:
# ==========================================
# 🌌 THE BRIDGE: Coroutines & send()
# ==========================================

# 1. A Generator that RECEIVES data (Coroutine)
def data_consumer():
    print("Waiting for data...")
    while True:
        # yield pauses and WAITS for a value to be sent in
        value = yield 
        print(f"Consumed: {value}")

gen = data_consumer()
next(gen)       # Prime the generator (runs up to the first yield)
gen.send(10)    # Sends 10 into the generator
gen.send(20)    # Sends 20 into the generator

print("\n---" * 10)

# 2. yield from (Delegation)
def sub_generator():
    yield 1
    yield 2

def main_generator():
    yield 0
    yield from sub_generator()  # Delegates to sub-generator
    yield 3

print("yield from output:")
for val in main_generator():
    print(val, end=" ")


Waiting for data...
Consumed: 10
Consumed: 20

---
---
---
---
---
---
---
---
---
---
yield from output:
0 1 2 3 


## 📋 19. The Ultimate One-Page Cheat Sheet

| Concept | Definition | Key Method/Keyword |
| :--- | :--- | :--- |
| **Iterable** | Collection of items. | `__iter__()` |
| **Iterator** | Stateful object that produces values. | `__iter__()`, `__next__()` |
| **`iter()`** | Converts Iterable $\rightarrow$ Iterator. | Built-in `iter()` |
| **`next()`** | Fetches next value from Iterator. | Built-in `next()` |
| **`StopIteration`** | Exception raised when Iterator is empty. | Raised by `__next__()` |
| **Generator** | Function that acts as an Iterator. | `yield` |
| **Gen Expression** | Memory-efficient comprehension. | `(x for x in range())` |
| **Coroutine** | Generator that can receive data. | `gen.send(value)` |
| **Async/Await** | Native coroutine syntax. | `async def`, `await` |

### 🧠 Mental Models
* **Iterable** = DVD Box Set
* **Iterator** = DVD Player Bookmark
* **`yield`** = Save Game Checkpoint
* **`send()`** = Loading a Save State with new input
```



## 💡 Fun Facts & Summary

### 🎉 Fun Facts
1. 🐍 **The `for` loop is a lie:** Python doesn't actually have a native `for` loop at the C-level. It's essentially a `while` loop that catches `StopIteration`!
2. 🧠 **Generators save the Frame:** When a generator yields, Python literally keeps the function's stack frame alive in memory. It doesn't destroy it until the generator is garbage collected.
3. 🚀 **Async is just Iterators:** Under the hood, Python's `asyncio` event loop is just a massive engine that calls `__next__()` and `send()` on thousands of coroutine objects!
4. 📜 **History:** Generators were introduced in Python 2.3 (PEP 255). The `send()` method was added in Python 2.5 (PEP 342), turning them into full Coroutines.

### 🎯 Summary Checklist
- [x] **Iterable:** Has `__iter__()` (e.g., Lists).
- [x] **Iterator:** Has `__next__()` (e.g., `iter(list)`).
- [x] **`for` loop:** Calls `iter()`, then loops `next()` until `StopIteration`.
- [x] **Generator:** A function using `yield` that saves local state.
- [x] **Recursion:** Function calling itself. Needs Base Case + Memoization for speed.
- [x] **The Missing Link:** Iterators $\rightarrow$ Generators $\rightarrow$ Coroutines $\rightarrow$ Async/Await.

---
**🎊 Congratulations! You have mastered Python's Iteration Protocol, Generators, Recursion, and the bridge to Async/Await!**


| ITERATORS, GENERATORS & RECURSION REFERENCE |
| :--- | 
| **ITERATION** |
| 1. Iteration: Process of going through items one by one |
| 2. Iterable: Object with `__iter__()` method   |
| 3. Iterator: Object with `__iter__()` AND `__next__()` methods |
| 4. StopIteration: Signal that iteration is complete  |
|**GENERATORS**|
| 1. Generator: Function with 'yield' instead of 'return' |
| 2. Generator Expression: (x for x in range(10)) |
| 3. Lazy Evaluation: Values generated on-demand |
| 4. Memory Efficient: Only one value at a time |
|**RECURSION**|
| 1. Recursion: Function that calls itself |
| 2. Base Case: Stopping condition |
| 3. Decomposition: Break problem into smaller pieces |
| 4. Memoization: Caching results to avoid recalculation |
|**KEY PATTERNS**|
| 1. Iteration → For loops, while loops |
| 2. Generator → yield (can be used once) |
| 3. Recursion → Function calls itself (needs base case)|
| 4. Memoization → Cache results for optimization |
|**COMPLEXITIES**|
| 1. Fibonacci (naive): O(2ⁿ) |
| 2. Fibonacci (memoized): O(n) |
| 3. Factorial: O(n) |
| 4. Power Set: O(2ⁿ) |